# H3 V1.5 视频换人 · RTX 6000 96GB · 完整部署版

这一个 notebook 已包含完整安装、模型下载、工作流和 Cloudflare 打开逻辑，无需额外的修复脚本。保持连接你的 **96GB RTX 6000 GPU runtime**，依次运行前四步即可：

**1. 环境检测 → 2. 安装 ComfyUI → 3. 下载模型与工作流 → 4. 启动并打开 ComfyUI**。部署步骤无需填写参数。

与其他 notebook 一样，在 Colab 左侧 **🔑 Secrets** 设置 `CF_TUNNEL_TOKEN` 并开启访问权限，Cloudflare Tunnel 的 Service URL 使用 `http://127.0.0.1:8188`。第 4 步连接后会显示 **“点击打开 ComfyUI”** 域名链接，并报告外网检查结果。

同一次部署提供三份独立工作流：**01_H3_Single_Full_Body（单人全身换装）**、**02_H3_Face_Only（只换脸）**、**03_H3_Multi_People（可选 1–5 人分别替换）**。03 左上角下拉选择人数，选几人就显示几张参考图、运行几遍生成，未选槽无需上传。默认时长均为 **0（整段原视频）**。前四步部署共用，模型共用，无需另外下载人脸模型。文末可选单元格也可选择模式和人数。

只换脸与多人版是基于作者原生节点编写的扩展工作流，尚未做 GPU 效果验证；应先检查遮罩预览。只换脸使用现有 SAM 对 `face` 分割，多人版逐人生成并合成，不属于作者已验证的原始单人配方。

面向已有的 **96GB NVIDIA GPU Linux runtime**；建议至少 64GiB RAM、约 65GiB 可用磁盘及额外素材空间。首次下载约 43.1GiB 权重，同一 runtime 可重复使用。

**已有旧版安装**：在同一台 GPU 机器上运行本版即可，已有模型会校验并复用。旧版的 8189 端口服务会在空闲后自动切换到 8188。

### 1. 环境检测

检测 GPU、显存、系统内存与磁盘。直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 1. 环境检测 {display-mode: "form"}
# 控制器完整内嵌，下载单个 ipynb 即可运行。
RUNTIME_SOURCE = r'''"""Standard-library controller embedded in the standalone H3 Colab notebook.

Uses the pinned H3 Relay API recipe and native-node variants. Model execution remains in
ComfyUI's isolated Python environment; importing this module does no GPU work.
"""
from __future__ import annotations

import copy
import ast
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import signal
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from urllib.parse import urlsplit
import uuid

COMFY_REVISION = "79be670e2d9be63e238785af307369d2b9039ed1"
RELAY_REVISION = "e2a7665b6eb8ea37a88c0de2c9c6b69a8f7c9220"
H3_REVISION = "912b6e38b2092ec4d2e86d24183e0d59ea059d48"
SAM_REVISION = "7bb8374780a725b4353ed31f3a9395c9742b5621"
SWAP_REVISION = "96ea3407b230b6f7df71c58c70e31df96bd7e533"
TURBO_REVISION = "30f485ed874aa44c8183348ad361c5eabad318da"
CLOUDFLARED_VERSION = "2026.10.6"
CLOUDFLARED_SHA256 = "1d356dac769d0dbf62ebe588c4034824bbff98d5790c2f844cc99e0e2f57462d"
# Upstream video_prep.py caps the full prepared source at 3592 frames / 24fps.
# Keep a whole-second limit below its 149.66s UI maximum. Individual diffusion
# windows still use the author's 124-frame / 18-frame history recipe.
MAX_CLIP_SECONDS = 149.0
DEFAULT_MASK_EXPANSION = 96
# The standalone notebook appends its generated, compressed workflow bundle.
WORKFLOW_BUNDLE = {}
DEFAULT_PROMPT = (
    "Replace only the selected person in <Video 1> with the character shown in "
    "<Picture 1>, from head to toe. Match the reference character's face, hair, "
    "visual style and complete outfit, including garment lengths, skirt hemline, "
    "exposed legs and footwear. Completely remove the selected person's original "
    "clothing. Do not retain the original long skirt or trousers beneath the new "
    "outfit, and do not layer the reference outfit over the source clothing. "
    "Reconstruct the legs and visible skin as shown in the reference, and restore "
    "the surrounding background where the original clothing extended beyond the "
    "reference outfit's silhouette. Follow the source person's position, scale, "
    "pose, gaze, expression, movement and timing. Preserve the background outside "
    "the replaced person, camera movement, lighting, other people and source audio timing."
)

# Retain the upstream interface. Only retry empty real frames; never copy a
# neighbour's mask, skip frames, or invent a full-frame mask to hide a failure.
SAM_RECOVERY_METHOD = """    def recover(self, masks, images, model, conditioning, selection, source_frames, recover_empty_frames):
        import json
        import torch
        import comfy.model_management
        from comfy_extras.nodes_sam3 import SAM3_Detect
        if len(masks) < source_frames or len(images) < source_frames:
            raise ValueError("Tracking must cover the complete source timeline")
        result = masks.clone().cpu()
        empty = torch.where((result[:source_frames] > .5).flatten(1).sum(1) == 0)[0].tolist()
        recovered, thresholds_used, unresolved = [], {}, []
        if recover_empty_frames:
            for frame in empty:
                comfy.model_management.throw_exception_if_processing_interrupted()
                print(f"[H3 SAM] Recover frame {frame} ({frame / 24:.3f}s)", flush=True)
                for threshold in (.35, .20, .10):
                    detection = SAM3_Detect.execute(model=model, image=images[frame:frame+1],
                        conditioning=conditioning, threshold=threshold, refine_iterations=2, individual_masks=True)[0]
                    if not (detection.flatten(1).sum(1) > 4).any():
                        continue
                    result[frame] = SelectSubject().select(detection, **selection)[0][0].cpu()
                    recovered.append(frame)
                    thresholds_used[str(frame)] = threshold
                    break
                else:
                    unresolved.append(frame)
        else:
            unresolved = empty
        if unresolved:
            locations = ", ".join(f"{i} ({i / 24:.3f}s)" for i in unresolved[:20])
            raise ValueError(
                f"SAM could not recover the selected person at source frames: {locations}; "
                f"{len(unresolved)} missing / {source_frames} real frames. "
                "Recovery thresholds: 0.35, 0.20, 0.10. Check these moments for occlusion, "
                "an absent person or a cut; correct the SAM person description or split the shot. "
                "Frames have not been skipped or replaced with fabricated masks.")
        report = json.dumps({"empty_tracking_frames": empty, "recovered_frames": recovered,
                             "thresholds_used": thresholds_used, "unresolved_frames": unresolved})
        return {"result": (result, report), "ui": {"text": [report]}}
"""

# File sizes and SHA-256 values from the repositories' LFS metadata, 2026-11-02.
MODELS = [
    dict(repo="Comfy-Org/MiniMax-H3", revision=H3_REVISION, directory="",
         file="diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
         size=20970379616, sha256="9255f52b6677845ad238f20dfaafa94727053694127ab7f255c048f0f9365779"),
    dict(repo="Comfy-Org/MiniMax-H3", revision=H3_REVISION, directory="",
         file="text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
         size=15687142551, sha256="35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6"),
    dict(repo="Comfy-Org/MiniMax-H3", revision=H3_REVISION, directory="",
         file="vae/minimax_h3_video_vae_fp16.safetensors", size=5207808496,
         sha256="7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522"),
    dict(repo="Comfy-Org/MiniMax-H3", revision=H3_REVISION, directory="",
         file="vae/minimax_h3_audio_vae_fp32.safetensors", size=605254808,
         sha256="8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48"),
    dict(repo="Comfy-Org/sam3.1", revision=SAM_REVISION, directory="",
         file="checkpoints/sam3.1_multiplex_fp16.safetensors", size=1745546848,
         sha256="9ba99c92703c2e8b4f47de2d34a539bb8e18923049e238b780d70dbe6368eb03"),
    dict(repo="lightx2v/Minimax-h3-Turbo", revision=TURBO_REVISION, directory="loras",
         file="minimax_h3_fl2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors",
         size=1956193000, sha256="08cfe946033af7d27719b964b6e0a0e50c32138daabbd6ce4137e23df6bf9980"),
    dict(repo="akatz-ai/MiniMax-H3-Character-Swap-LoRA", revision=SWAP_REVISION,
         directory="loras", file="h3_character_swap_v1.5.safetensors", size=155110792,
         sha256="8f2c3c3eba161df8d0c6099598b4730f641f5ce01e04cd16d2427cff8a35218f"),
]


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n")
    temporary.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(chunk)
    return digest.hexdigest()


def restore_sam_recovery(relay):
    """Undo only our byte-for-byte managed edit before a pinned git checkout."""
    target = Path(relay) / "h3_relay/mask_tools.py"
    record = target.with_suffix(".h3-recovery.json")
    backup = target.with_suffix(".h3-recovery.original")
    if not record.is_file():
        return
    state = json.loads(record.read_text())
    current = sha256_file(target)
    if current == state["original_sha256"]:
        return
    if current != state["patched_sha256"] or sha256_file(backup) != state["original_sha256"]:
        raise RuntimeError("SAM 补检文件还有其他修改，已保留原文件，请先检查 mask_tools.py。")
    target.write_bytes(backup.read_bytes())


def patch_sam_recovery(relay):
    target = Path(relay) / "h3_relay/mask_tools.py"
    record = target.with_suffix(".h3-recovery.json")
    backup = target.with_suffix(".h3-recovery.original")
    if record.is_file():
        state = json.loads(record.read_text())
        if sha256_file(target) == state["patched_sha256"]:
            return
        if sha256_file(target) != state["original_sha256"]:
            raise RuntimeError("SAM 补检文件与已记录版本不一致，已保留文件，请检查 mask_tools.py。")
    source = target.read_text()
    tree = ast.parse(source)
    cls = next(n for n in tree.body if isinstance(n, ast.ClassDef) and n.name == "RecoverMasks")
    method = next(n for n in cls.body if isinstance(n, ast.FunctionDef) and n.name == "recover")
    lines = source.splitlines(keepends=True)
    original = "".join(lines[method.lineno - 1:method.end_lineno])
    if "threshold=.35" not in original or "SelectSubject().select" not in original:
        raise RuntimeError("SAM 补检接口已变化，未修改源码；请使用匹配版本的 H3 Relay。")
    updated = "".join(lines[:method.lineno - 1]) + SAM_RECOVERY_METHOD + "".join(lines[method.end_lineno:])
    compile(updated, str(target), "exec")
    original_sha256 = sha256_file(target)
    backup.write_bytes(target.read_bytes())
    target.write_text(updated)
    write_json(record, dict(original_sha256=original_sha256, patched_sha256=sha256_file(target)))


def download_weights(root, workers=2):
    """Called in the venv, so the Colab kernel's packages stay untouched."""
    from concurrent.futures import ThreadPoolExecutor
    from huggingface_hub import hf_hub_download

    model_dir = Path(root) / "ComfyUI/models"

    def download_one(item):
        base = model_dir / item["directory"]
        target = base / item["file"]
        if target.is_file() and target.stat().st_size == item["size"]:
            if sha256_file(target) == item["sha256"]:
                print("[已校验] " + item["file"], flush=True)
                return
        print("[下载] " + item["file"], flush=True)
        result = Path(hf_hub_download(
            repo_id=item["repo"], filename=item["file"], revision=item["revision"],
            local_dir=str(base), force_download=target.exists(),
        ))
        if result != target or target.stat().st_size != item["size"]:
            raise RuntimeError("下载大小不符：" + str(target))
        if sha256_file(target) != item["sha256"]:
            raise RuntimeError("SHA-256 校验失败，请重新运行部署：" + str(target))
        print("[已校验] " + item["file"], flush=True)

    with ThreadPoolExecutor(max_workers=max(1, min(int(workers), 3))) as pool:
        # Consume every result: a failed download must fail the deployment.
        list(pool.map(download_one, MODELS))


def probe_video(path, count_frames=False):
    command = ["ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json"]
    if count_frames:
        command.append("-count_frames")
    command.append(str(path))
    data = json.loads(subprocess.check_output(command, timeout=120))
    video = next((s for s in data["streams"] if s["codec_type"] == "video"), None)
    if video is None:
        raise ValueError("文件中没有可解码的视频流。")
    duration = video.get("duration") or data.get("format", {}).get("duration")
    if not duration or not math.isfinite(float(duration)) or float(duration) <= 0:
        raise ValueError("无法读取视频时长，请转换为普通 MP4 后重试。")
    return dict(duration=float(duration), width=video["width"], height=video["height"],
                fps=video.get("avg_frame_rate", "0/1"),
                frames=video.get("nb_read_frames") or video.get("nb_frames"),
                has_audio=any(s["codec_type"] == "audio" for s in data["streams"]))


def validate_options(start, duration, resolution, selection, index, expansion, seed):
    if not all(math.isfinite(float(x)) for x in (start, duration)):
        raise ValueError("起点和时长必须是有限数值。")
    if start < 0 or not 0 <= duration <= MAX_CLIP_SECONDS:
        raise ValueError("起点须 ≥ 0；时长须为 0–149 秒，0 表示处理整段（最多 149 秒）。")
    if str(resolution) not in {"480", "768"}:
        raise ValueError("分辨率档位须为 480 或 768。")
    if selection not in {"largest", "leftmost", "rightmost", "index"}:
        raise ValueError("人物选择须为 largest / leftmost / rightmost / index。")
    if type(index) is not int or index < 0 or type(expansion) is not int or not 0 <= expansion <= 256:
        raise ValueError("人物序号须 ≥ 0，遮罩扩张须为 0–256 的整数。")
    if type(seed) is not int or not 0 <= seed < 2**64:
        raise ValueError("种子须为 0–2^64-1 的整数。")


def make_graph(template, video, reference, *, subject="person", prompt=DEFAULT_PROMPT,
               resolution="768", duration=0.0, selection="largest", index=0,
               expansion=DEFAULT_MASK_EXPANSION, seed=42):
    validate_options(0, duration, resolution, selection, index, expansion, seed)
    if not subject.strip() or not prompt.strip():
        raise ValueError("人物描述和换人提示词不能为空。")
    graph = copy.deepcopy(template)
    graph.pop("mask_preview", None)  # Use a playable mask video instead of hundreds of PNGs.
    graph["source"]["inputs"]["file"] = video
    graph["reference"]["inputs"]["image"] = reference
    graph["prepare"]["inputs"].update(resolution=str(resolution), duration_seconds=float(duration))
    graph["subject"]["inputs"]["text"] = subject.strip()
    graph["select"]["inputs"].update(selection=selection, index=index)
    graph["detect"]["inputs"]["threshold"] = 0.20
    graph["track"]["inputs"]["detection_threshold"] = 0.20
    graph["expand_mask"]["inputs"]["expansion_px"] = expansion
    graph["plan"]["inputs"].update(seed=seed, window_frames="124", history_frames="18")
    graph["reference_conditioning"]["inputs"]["prompt"] = prompt.strip()
    graph["adapter"]["inputs"].update(lora_name="h3_character_swap_v1.5.safetensors", strength_model=1.0)
    graph["turbo"]["inputs"]["strength_model"] = 1.0
    graph["schedule"]["inputs"].update(steps=8, scheduler="normal", denoise=1.0)
    graph["guider"]["inputs"]["cfg"] = 1.0
    graph["sampler"]["inputs"]["sampler_name"] = "er_sde"
    graph["save"]["inputs"]["filename_prefix"] = "h3-character-swap/result"
    return graph


def configure_canvas(template):
    """Apply the same outfit/mask/full-duration defaults to the original canvas.

    Keep every connection, layout, input filename, selection and sampler intact.
    The named widget metadata is the pinned author's positional widget schema.
    """
    canvas = copy.deepcopy(template)
    defaults = {
        "adapter": {"lora_name": "h3_character_swap_v1.5.safetensors"},
        "prepare": {"duration_seconds": 0.0},
        "expand_mask": {"expansion_px": DEFAULT_MASK_EXPANSION},
        "detect": {"threshold": 0.20},
        "track": {"detection_threshold": 0.20},
        "reference_conditioning": {"prompt": DEFAULT_PROMPT},
    }
    found = set()
    for node in canvas["nodes"]:
        updates = defaults.get(node["id"], {})
        if not updates:
            continue
        named = node.get("widgets_values_named", {})
        for key, value in updates.items():
            if key not in named:
                raise ValueError(f"工作流节点 {node['id']} 缺少 {key}，请使用匹配版本的工作流。")
            slot = list(named).index(key)
            node["widgets_values"][slot] = value
            named[key] = value
        found.add(node["id"])
    if found != defaults.keys():
        raise ValueError("工作流缺少换人、时长或遮罩节点，请使用匹配版本的工作流。")
    return canvas


def make_mode_graph(template, mode, video, reference, *, reference_b=None,
                    duration=0.0, resolution="768", seed=42, subject=None,
                    index_a=0, index_b=1, references=None, people_count=2,
                    person_indices=None):
    legacy_two = mode == "two_people"
    if legacy_two:
        mode = "multi_people"
    if mode not in {"single_full", "face_only", "multi_people"}:
        raise ValueError("请选择 single_full / face_only / multi_people。")
    validate_options(0, duration, resolution, "index", index_a, 8 if mode == "face_only" else 96, seed)
    if mode == "multi_people":
        if type(people_count) is not int or not 1 <= people_count <= 5:
            raise ValueError("替换人数须为 1–5 的整数。")
        if references is None:
            references = [reference] + ([reference_b] if reference_b else [])
        if len(references) != people_count or any(not isinstance(value, str) or not value.strip() for value in references):
            raise ValueError("所选人数必须与实际上传的参考图数量一致。")
        if template.get("people_count", {}).get("inputs", {}).get("value") != people_count:
            raise ValueError("人数与工作流分支不一致，请取得对应人数的模板。")
        indices = list(person_indices) if person_indices is not None else (
            [index_a, index_b] if legacy_two else list(range(people_count)))
        if len(indices) != people_count or any(type(value) is not int or not 0 <= value < 5 for value in indices) or len(set(indices)) != people_count:
            raise ValueError("人物编号须为 0–4，数量匹配且不能重复。")
    graph = copy.deepcopy(template)
    for key in list(graph):
        if key == "mask_preview" or key.endswith("_mask_preview"):
            graph.pop(key)
    graph["source"]["inputs"]["file"] = video
    graph["reference"]["inputs"]["image"] = reference
    graph["prepare"]["inputs"]["resolution"] = str(resolution)
    if mode == "multi_people":
        graph["duration_control"]["inputs"]["value"] = float(duration)
    else:
        graph["prepare"]["inputs"]["duration_seconds"] = float(duration)
    graph["plan"]["inputs"]["seed"] = seed
    if subject is not None:
        if not subject.strip():
            raise ValueError("SAM 描述不能为空。")
        text = subject.strip()
        graph["subject"]["inputs"]["text"] = text + ":5" if mode == "multi_people" and ":" not in text else text
    if mode == "multi_people":
        for i, letter in enumerate("abcde"[:people_count]):
            prefix = "" if i == 0 else letter + "_"
            graph["reference" if i == 0 else "reference_" + letter]["inputs"]["image"] = references[i]
            graph[prefix + "mask"]["inputs"]["object_indices"] = str(indices[i])
            graph[prefix + "plan"]["inputs"]["seed"] = (seed + i) % 2**64
    return graph


def install_workflow_bundle(comfy, bundle):
    directory = Path(comfy) / "user/default/workflows"
    paths = []
    for profile in bundle.values():
        if profile.get("web_assets"):
            web = Path(comfy) / "custom_nodes/h3-relay/web"
            if not web.parent.is_dir():
                raise RuntimeError("找不到现有 h3-relay 节点目录，请确认原部署目录。")
            web.mkdir(parents=True, exist_ok=True)
            current_scripts = {name for name in profile["web_assets"] if name.endswith(".js")}
            for old in web.glob("h3_colab_people_selector_*.js"):
                if old.name not in current_scripts:
                    old.rename(old.with_name(old.name + ".disabled-" + uuid.uuid4().hex[:8]))
            for name, source in profile["web_assets"].items():
                if Path(name).name != name or not name.startswith("h3_colab_people_"):
                    raise ValueError("人数控件资源文件名不合法。")
                target = web / name
                temporary = target.with_name(target.name + ".tmp")
                temporary.write_text(source, encoding="utf-8")
                temporary.replace(target)
        path = directory / (profile["filename"] + ".json")
        if path.is_file():
            if json.loads(path.read_text()) == profile["canvas"]:
                paths.append(str(path))
                continue
            backup = path.with_name(path.name + ".backup-" + uuid.uuid4().hex[:8])
            shutil.copy2(path, backup)
        write_json(path, profile["canvas"])
        paths.append(str(path))
    return paths


def ancestors(graph, output):
    """Keep the SAM-only dependency subgraph for a cheap pre-generation preview."""
    keep = set()

    def visit(key):
        if key in keep:
            return
        keep.add(key)
        for value in graph[key]["inputs"].values():
            if isinstance(value, list) and len(value) == 2 and isinstance(value[0], str):
                if value[0] in graph and type(value[1]) is int:
                    visit(value[0])

    visit(output)
    return {key: copy.deepcopy(node) for key, node in graph.items() if key in keep}


def preview_graph(graph, target="a"):
    if "people_count" in graph or "b_blend_trim" in graph:
        if target not in "abcde" or target + "_blend_trim" not in graph:
            raise ValueError("请选择当前工作流实际存在的人物 A–E。")
        mask_output = target + "_blend_trim"
        images = [mask_output, 0]
    elif "face_trim" in graph:
        mask_output, images = "face_trim", ["face_trim", 0]
    else:
        mask_output, images = "expand_mask", ["expand_mask", 1]
    preview = ancestors(graph, mask_output)
    preview["mask_video"] = dict(class_type="CreateVideo", inputs={
        "images": images, "fps": 24.0, "audio": ["prepare", 1]})
    preview["save"] = copy.deepcopy(graph["save"])
    prefix = "h3-character-swap/mask" + ("-" + target if "people_count" in graph or "b_blend_trim" in graph else "")
    preview["save"]["inputs"].update(video=["mask_video", 0], filename_prefix=prefix)
    return preview


def find_saved_video(history, output_dir):
    """Accept the pinned SaveVideo UI payload without assuming its media key."""
    root = Path(output_dir).resolve()

    def walk(value):
        if isinstance(value, dict):
            name = value.get("filename")
            if isinstance(name, str) and name.lower().endswith(".mp4") and value.get("type") == "output":
                candidate = (root / value.get("subfolder", "") / name).resolve()
                if candidate.is_relative_to(root) and candidate.is_file():
                    return candidate
            for child in value.values():
                found = walk(child)
                if found:
                    return found
        elif isinstance(value, list):
            for child in value:
                found = walk(child)
                if found:
                    return found
        return None

    found = walk(history.get("outputs", {}).get("save", {}))
    if not found:
        raise RuntimeError("任务结束但 SaveVideo 未返回有效 MP4，请查看运行记录和服务日志。")
    return found


def cloudflare_routes(log_text):
    """Read the named tunnel's actual ingress; never invent a public URL."""
    decoder = json.JSONDecoder()
    routes = []
    for line in log_text.splitlines():
        if "Updated to new configuration" not in line or "config=" not in line:
            continue
        try:
            value, _ = decoder.raw_decode(line.split("config=", 1)[1])
            config = json.loads(value) if isinstance(value, str) else value
            routes = [r for r in config.get("ingress", []) if isinstance(r, dict)]
        except (ValueError, TypeError, AttributeError):
            continue
    return routes


def matching_cloudflare_url(routes, port):
    for route in routes:
        host = route.get("hostname", "")
        try:
            service = urlsplit(route.get("service", ""))
            if (re.fullmatch(r"[A-Za-z0-9](?:[A-Za-z0-9.-]*[A-Za-z0-9])?", host)
                    and service.hostname in {"localhost", "127.0.0.1", "::1"}
                    and service.port == port and service.scheme == "http"):
                return "https://" + host
        except ValueError:
            continue
    return None


def can_bind_port(port):
    # Match aiohttp/asyncio's Unix listener: closing HTTP connections can remain
    # in TIME_WAIT after the process exits. A plain bind gives a false conflict.
    with socket.socket() as connection:
        connection.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            connection.bind(("127.0.0.1", int(port)))
        except OSError:
            return False
    return True


class H3Colab:
    def __init__(self, root="/content/H3CharacterSwap", port=8188):
        self.root = Path(root).expanduser().resolve()
        self.comfy = self.root / "ComfyUI"
        self.relay = self.comfy / "custom_nodes/h3-relay"
        self.python = self.root / "venv/bin/python"
        self.port = int(port)
        if not 1024 <= self.port <= 65535:
            raise ValueError("端口须为 1024–65535 的整数。")
        self.base = f"http://127.0.0.1:{self.port}"
        self.log = self.root / "comfyui.log"
        self.root.mkdir(parents=True, exist_ok=True)
        self.env = dict(os.environ, PYTHONUNBUFFERED="1", HF_HOME=str(self.root / "hf-cache"),
                        HF_XET_CHUNK_CACHE_SIZE_BYTES="0", TOKENIZERS_PARALLELISM="false")
        # These venv processes must not inherit the Colab kernel's Python path.
        self.env.pop("PYTHONPATH", None)
        self.env.pop("PYTHONHOME", None)

    def request(self, route, body=None, timeout=30):
        encoded = None if body is None else json.dumps(body).encode()
        req = urllib.request.Request(self.base + route, data=encoded,
                                     headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=timeout) as response:
                return json.load(response)
        except urllib.error.HTTPError as error:
            detail = error.read().decode(errors="replace")
            raise RuntimeError(f"ComfyUI {route} HTTP {error.code}: {detail[:6000]}") from error

    def run(self, args, *, cwd=None, visible=False):
        if visible:
            subprocess.run([str(a) for a in args], cwd=cwd, env=self.env, check=True)
            return
        with (self.root / "install.log").open("a") as log:
            result = subprocess.run([str(a) for a in args], cwd=cwd, env=self.env,
                                    stdout=log, stderr=subprocess.STDOUT)
        if result.returncode:
            tail = (self.root / "install.log").read_text(errors="replace")[-6000:]
            raise RuntimeError(f"安装命令失败（{result.returncode}）：\n{tail}")

    def checkout(self, directory, repo, revision):
        fresh = not directory.exists()
        if fresh:
            self.run(["git", "clone", "--depth", "1", repo, directory])
        origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=directory, text=True).strip()
        if origin.removesuffix(".git") != repo.removesuffix(".git"):
            raise RuntimeError("目录已被其他仓库占用，请更换安装目录：" + str(directory))
        current = subprocess.run(["git", "rev-parse", "HEAD"], cwd=directory, capture_output=True, text=True)
        index_name = subprocess.check_output(["git", "rev-parse", "--git-path", "index"],
                                             cwd=directory, text=True).strip()
        index_path = Path(index_name)
        if not index_path.is_absolute():
            index_path = directory / index_path
        # The old --no-checkout clone has HEAD but no index or working files.
        # Do not mistake that unpopulated state for staged user deletions.
        if not fresh and index_path.exists():
            if directory == self.relay:
                restore_sam_recovery(directory)
            self.run(["git", "diff", "--exit-code"], cwd=directory)
            self.run(["git", "diff", "--cached", "--exit-code"], cwd=directory)
        if current.stdout.strip() != revision:
            self.run(["git", "fetch", "--depth", "1", "origin", revision], cwd=directory)
        # Always materialize the requested tree, including when HEAD already matches.
        self.run(["git", "checkout", "--detach", revision], cwd=directory)

    def ensure_relay_sources(self):
        required = ("__init__.py", "h3_relay/toolkit.py", "h3_relay/mask_tools.py",
                    "example_workflows/toolkit/H3-Character-Swap.json",
                    "example_workflows/toolkit/H3-Character-Swap.api.json")
        if any(not (self.relay / name).is_file() for name in required):
            self.checkout(self.relay, "https://github.com/akatz-ai/h3-relay.git", RELAY_REVISION)
        missing = [name for name in required if not (self.relay / name).is_file()]
        if missing:
            raise RuntimeError("H3 Relay 检出不完整，尚未开始下载模型。缺少：" + ", ".join(missing))
        patch_sam_recovery(self.relay)

    def check_environment(self):
        if sys.platform != "linux" or not shutil.which("nvidia-smi"):
            raise RuntimeError("请在已连接 NVIDIA GPU 的 Linux Colab runtime 中运行。")
        gpu = subprocess.check_output([
            "nvidia-smi", "-i", "0", "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader,nounits"], text=True).strip()
        print("GPU / MiB / 驱动：", gpu)
        memory_mib = float(gpu.split(",")[-2].strip())
        if memory_mib < 80_000:
            raise RuntimeError("本配置面向单卡 96GB RTX 6000，请连接相应 GPU runtime。")
        ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
        if ram < 48:
            raise RuntimeError(f"系统内存仅 {ram:.1f} GiB，建议使用至少 64GiB RAM 的 GPU 机器。")
        remaining = sum(m["size"] for m in MODELS if not (
            (p := self.comfy / "models" / m["directory"] / m["file"]).is_file()
            and p.stat().st_size == m["size"]))
        if shutil.disk_usage(self.root).free < remaining + 20 * 1024**3:
            raise RuntimeError(f"磁盘不足：尚需权重 {remaining / 1024**3:.1f} GiB，另留 20GiB 安装与输出空间。")
        print(f"系统内存 {ram:.1f} GiB；核心版本与模型 revision 已固定。")
        print("[OK] 环境检测完成。")

    def install_environment(self):
        print("准备 FFmpeg 与固定版本源码…", flush=True)
        if not all(shutil.which(x) for x in ("git", "ffmpeg", "ffprobe")):
            self.run(["apt-get", "update", "-qq"])
            self.run(["apt-get", "install", "-y", "-qq", "git", "ffmpeg"])
        self.checkout(self.comfy, "https://github.com/Comfy-Org/ComfyUI.git", COMFY_REVISION)
        self.checkout(self.relay, "https://github.com/akatz-ai/h3-relay.git", RELAY_REVISION)
        self.ensure_relay_sources()
        print("创建独立 Python 3.12 环境…", flush=True)
        self.run([sys.executable, "-m", "pip", "install", "-q", "uv"])
        if not self.python.is_file():
            # uv supplies Python 3.12 even when the hosted notebook kernel changes.
            self.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", self.root / "venv"])
        self.run([self.python, "-c", "import sys; assert sys.version_info[:2] == (3, 12), '需要 Python 3.12'"])
        print("安装 PyTorch 2.11.0 / cu128 与 H3 依赖…", flush=True)
        self.run([sys.executable, "-m", "uv", "pip", "install", "--python", self.python,
                  "--index-url", "https://download.pytorch.org/whl/cu128",
                  "torch==2.11.0", "torchvision==0.26.0", "torchaudio==2.11.0"])
        constraints = self.root / "constraints.txt"
        constraints.write_text("torch==2.11.0+cu128\ntorchvision==0.26.0+cu128\ntorchaudio==2.11.0+cu128\ntransformers==4.57.6\n")
        self.run([sys.executable, "-m", "uv", "pip", "install", "--python", self.python,
                  "--extra-index-url", "https://download.pytorch.org/whl/cu128",
                  "--index-strategy", "unsafe-best-match",
                  "-c", constraints, "-r", self.comfy / "requirements.txt",
                  "torchaudio==2.11.0", "huggingface_hub[hf_xet]>=0.34,<1"])
        self.run([self.python, "-c", (
            "import torch, importlib.metadata as m; "
            "assert torch.version.cuda == '12.8', '需要 PyTorch cu128 构建'; "
            "assert torch.cuda.is_available(), 'CUDA 不可用，请检查驱动'; "
            "assert torch.cuda.is_bf16_supported(), 'GPU 必须支持 BF16'; "
            "x=torch.ones((32,32),device='cuda',dtype=torch.bfloat16); x@x; torch.cuda.synchronize(); "
            "print('CUDA 检查通过:',torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0)); "
            "assert m.version('comfy-kitchen') == '0.2.35'; "
            "assert m.version('comfyui-frontend-package') == '1.53.6'"
        )], visible=True)
        self.ensure_cloudflared()
        print("[OK] ComfyUI 与 H3 Relay 安装完成。")

    def prepare_models(self, token="", workers=2):
        self.ensure_relay_sources()
        print("下载并校验 7 个模型（约 43.1GiB，支持复用已完成文件）…", flush=True)
        if token:
            self.env["HF_TOKEN"] = token
        try:
            self.run([self.python, "-c", (
                "import sys; sys.path.insert(0,sys.argv[1]); "
                "from h3_colab import download_weights; download_weights(sys.argv[1],int(sys.argv[2]))"
            ), self.root, str(workers)], visible=True)
        finally:
            self.env.pop("HF_TOKEN", None)
        workflow_dir = self.comfy / "user/default/workflows"
        workflow_dir.mkdir(parents=True, exist_ok=True)
        if WORKFLOW_BUNDLE:
            install_workflow_bundle(self.comfy, WORKFLOW_BUNDLE)
        else:
            canvas = configure_canvas(json.loads(
                (self.relay / "example_workflows/toolkit/H3-Character-Swap.json").read_text()))
            write_json(workflow_dir / "H3-Character-Swap-V1.5.json", canvas)
        write_json(self.root / "model-manifest.json", MODELS)
        print("[OK] 模型与换人工作流准备完成。")

    def install(self, token="", workers=2):
        self.check_environment()
        self.install_environment()
        self.prepare_models(token=token, workers=workers)
        self.start()
        print("[OK] 部署完成。")

    def start(self):
        self.ensure_relay_sources()
        record = self.root / "server.json"
        launcher = self.root / "comfy_launcher.py"
        entry = launcher if launcher.is_file() else self.comfy / "main.py"
        args = [str(self.python), str(entry), "--listen", "127.0.0.1",
                "--port", str(self.port), "--disable-auto-launch", "--vram-headroom", "6",
                "--enable-cors-header", "--enable-compress-response-body",
                "--disable-all-custom-nodes", "--whitelist-custom-nodes", "h3-relay"]
        signature = hashlib.sha256((sha256_file(entry) + sha256_file(
            self.relay / "h3_relay/mask_tools.py")).encode()).hexdigest()
        if record.is_file():
            old = json.loads(record.read_text())
            old_args = old.get("args", [])
            if old_args != args or old.get("entry_sha256") != signature:
                self.stop_owned_server(old)
        try:
            self.request("/system_stats", timeout=3)
            old = json.loads(record.read_text()) if record.exists() else {}
            if old.get("args") != args or not self.owns_process(old.get("pid", 0),
                                                               [self.comfy / "main.py", launcher]):
                raise RuntimeError(f"端口 {self.port} 已被其他进程使用，请更改 notebook 的端口。")
            print("复用本 notebook 已启动的 ComfyUI。")
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if not can_bind_port(self.port):
                raise RuntimeError(f"端口 {self.port} 仍有服务监听；未启动重复服务，请检查现有进程。")
            with self.log.open("a") as log:
                process = subprocess.Popen(args, cwd=self.comfy, env=self.env, stdin=subprocess.DEVNULL,
                                           stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
            write_json(record, {"pid": process.pid, "args": args, "entry_sha256": signature})
            deadline = time.monotonic() + 600
            while time.monotonic() < deadline:
                if process.poll() is not None:
                    raise RuntimeError("ComfyUI 启动失败：\n" + self.log.read_text(errors="replace")[-6000:])
                try:
                    self.request("/system_stats", timeout=3)
                    break
                except (urllib.error.URLError, TimeoutError, ConnectionError):
                    time.sleep(2)
            else:
                process.terminate()
                raise TimeoutError("ComfyUI 10 分钟内未就绪，请查看 " + str(self.log))
        info = self.request("/object_info", timeout=120)
        recipes = [profile["api"] for profile in WORKFLOW_BUNDLE.values()] if WORKFLOW_BUNDLE else [self.template()]
        required = {node["class_type"] for recipe in recipes for node in recipe.values()}
        missing = required - set(info)
        if missing:
            raise RuntimeError("缺少工作流节点：" + ", ".join(sorted(missing)) + "\n" + self.log.read_text(errors="replace")[-4000:])
        self.object_info = info

    def configure_launcher(self, source):
        source = source.replace('Path("/content/ComfyUI")', f"Path({str(self.comfy)!r})")
        compile(source, "comfy_launcher.py", "exec")
        (self.root / "comfy_launcher.py").write_text(source)

    def owns_process(self, pid, entries):
        try:
            command = Path(f"/proc/{int(pid)}/cmdline").read_bytes().split(b"\0")
            return any(str(entry).encode() in command for entry in entries)
        except (OSError, ValueError):
            return False

    def stop_owned_server(self, record):
        pid = record.get("pid", 0)
        if not self.owns_process(pid, [self.comfy / "main.py", self.root / "comfy_launcher.py"]):
            return
        args = record.get("args", [])
        try:
            port = int(args[args.index("--port") + 1])
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/queue", timeout=5) as response:
                queue = json.load(response)
        except Exception as error:
            raise RuntimeError("旧服务状态无法确认，请先检查 comfyui.log 后重跑第 4 步。") from error
        if queue.get("queue_running") or queue.get("queue_pending"):
            raise RuntimeError("旧服务仍在执行任务，完成后重跑第 4 步即可更新访问方式。")
        print("正在更新本 notebook 的服务访问配置…", flush=True)
        os.kill(pid, signal.SIGTERM)
        for _ in range(60):
            exited = not self.owns_process(pid, [self.comfy / "main.py", self.root / "comfy_launcher.py"])
            if exited and can_bind_port(port):
                return
            time.sleep(0.5)
        raise RuntimeError(f"旧服务或其监听端口 {port} 尚未释放；未启动重复服务，请检查 comfyui.log。")

    def ensure_cloudflared(self):
        binary = self.root / "bin/cloudflared"
        if binary.is_file() and sha256_file(binary) == CLOUDFLARED_SHA256:
            return binary
        binary.parent.mkdir(parents=True, exist_ok=True)
        temporary = binary.with_suffix(".download")
        url = (f"https://github.com/cloudflare/cloudflared/releases/download/"
               f"{CLOUDFLARED_VERSION}/cloudflared-linux-amd64")
        print("准备 Cloudflare Tunnel…", flush=True)
        with urllib.request.urlopen(url, timeout=120) as response, temporary.open("wb") as target:
            shutil.copyfileobj(response, target)
        if sha256_file(temporary) != CLOUDFLARED_SHA256:
            raise RuntimeError("cloudflared 校验失败，请重新运行本步骤。")
        temporary.chmod(0o755)
        temporary.replace(binary)
        return binary

    def open_ui(self):
        token = os.environ.get("CF_TUNNEL_TOKEN", "").strip()
        if not token:
            try:
                from google.colab import userdata
                token = (userdata.get("CF_TUNNEL_TOKEN") or "").strip()
            except Exception:
                pass
        if not token:
            raise RuntimeError("请与其他 notebook 一样，在 Colab 左侧 🔑 Secrets 设置 CF_TUNNEL_TOKEN 并开启访问权限。")
        token = token.split()[-1]
        self.start()
        binary = self.ensure_cloudflared()
        log_path = self.root / "cloudflared.log"
        record_path = self.root / "cloudflared.json"
        if record_path.is_file():
            previous = json.loads(record_path.read_text())
            if self.owns_process(previous.get("pid", 0), [binary]):
                os.kill(previous["pid"], signal.SIGTERM)
        env = dict(self.env, TUNNEL_TOKEN=token)
        with log_path.open("w") as log:
            process = subprocess.Popen([
                str(binary), "tunnel", "--no-autoupdate", "--protocol", "http2",
                "--edge-ip-version", "4", "run", "--url", self.base,
            ], env=env, stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT,
               start_new_session=True)
        token = ""
        env.pop("TUNNEL_TOKEN", None)
        write_json(record_path, {"pid": process.pid})
        print("正在连接 Cloudflare，成功后会显示打开链接…", flush=True)
        deadline = time.monotonic() + 60
        url = None
        routes = []
        while time.monotonic() < deadline:
            if process.poll() is not None:
                raise RuntimeError("Cloudflare 连接失败，请检查 CF_TUNNEL_TOKEN。日志：" + str(log_path))
            log = log_path.read_text(errors="replace")
            routes = cloudflare_routes(log)
            url = matching_cloudflare_url(routes, self.port)
            if "Registered tunnel connection" in log and url:
                break
            time.sleep(1)
        else:
            if routes and not url:
                raise RuntimeError(f"Cloudflare 域名未指向 {self.base}，请检查 Tunnel 的 Service URL。")
            raise RuntimeError("Cloudflare 未返回可打开的域名，请查看 " + str(log_path))
        # The tunnel and local service are ready. Edge rules may reject a
        # notebook's HTTP probe while allowing the signed-in browser.
        print("[已连接] ComfyUI 访问地址：", url, flush=True)
        access = {"url": url, "tunnel_connected": True, "public_verified": False}
        write_json(self.root / "public-url.json", access)
        probe = urllib.request.Request(url + "/system_stats", headers={
            "User-Agent": "H3-Colab/1.0 (+https://github.com/boltguo/colab)",
            "Accept": "application/json",
        })
        last_error = None
        for attempt in range(3):
            try:
                with urllib.request.urlopen(probe, timeout=5) as response:
                    data = json.load(response)
                if not isinstance(data, dict) or "devices" not in data:
                    raise ValueError("域名未返回 ComfyUI 状态。")
                access.update(public_verified=True, http_status=200)
                write_json(self.root / "public-url.json", access)
                print("[OK] ComfyUI 公网访问正常：", url)
                return url
            except urllib.error.HTTPError as error:
                last_error = f"HTTP {error.code}"
                access["http_status"] = error.code
                if error.code in {401, 403}:
                    print(f"[请在浏览器确认] 外网探测返回 HTTP {error.code}，下方仍会显示访问链接。", flush=True)
                    break
            except Exception as error:
                last_error = str(error)
            if attempt < 2:
                time.sleep(2)
        access["public_probe_error"] = str(last_error)
        write_json(self.root / "public-url.json", access)
        print(f"[外网检查未通过] {last_error}。请点击链接确认；若浏览器也报错，请检查 Cloudflare 访问日志。")
        return url

    def template(self, mode="single_full", people_count=2):
        if mode == "two_people":
            mode = "multi_people"
        if WORKFLOW_BUNDLE:
            if mode not in WORKFLOW_BUNDLE:
                raise ValueError("未知工作流模式：" + str(mode))
            profile = WORKFLOW_BUNDLE[mode]
            if mode == "multi_people":
                if type(people_count) is not int or not 1 <= people_count <= 5:
                    raise ValueError("替换人数须为 1–5 的整数。")
                profile = profile["variants"][str(people_count)]
            return copy.deepcopy(profile["api"])
        if mode != "single_full":
            raise RuntimeError("当前旧控制器没有三模式工作流，请使用最新完整 notebook 或导入对应 JSON。")
        self.ensure_relay_sources()
        return json.loads((self.relay / "example_workflows/toolkit/H3-Character-Swap.api.json").read_text())

    def prepare_reference(self, reference):
        if not Path(reference).is_file():
            raise FileNotFoundError("找不到参考图：" + str(reference))
        target_image = self.comfy / "input" / ("reference_" + uuid.uuid4().hex[:12] + ".png")
        target_image.parent.mkdir(parents=True, exist_ok=True)
        self.run([self.python, "-c", (
            "import sys; from PIL import Image,ImageOps; "
            "im=ImageOps.exif_transpose(Image.open(sys.argv[1])).convert('RGB'); "
            "im.thumbnail((2048,2048)); im.save(sys.argv[2])"
        ), reference, target_image])
        return target_image.name

    def prepare_inputs(self, video, reference, start=0.0, duration=0.0):
        for path in (video, reference):
            if not Path(path).is_file():
                raise FileNotFoundError("找不到素材：" + str(path))
        validate_options(start, duration, "768", "largest", 0, DEFAULT_MASK_EXPANSION, 42)
        source = probe_video(video)
        if start >= source["duration"]:
            raise ValueError("起点已超过原视频时长。")
        selected = min(duration or source["duration"] - start, source["duration"] - start)
        if selected < 0.5 or selected > MAX_CLIP_SECONDS + 0.001:
            raise ValueError("本原生工作流每次处理 0.5–149 秒；更长素材请按镜头分段。")
        name = uuid.uuid4().hex[:12]
        inputs = self.comfy / "input"
        inputs.mkdir(parents=True, exist_ok=True)
        if start > 0:
            target_video = inputs / f"source_{name}.mkv"
            self.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y",
                      "-ss", str(start), "-i", video, "-t", str(selected),
                      "-map", "0:v:0", "-map", "0:a:0?", "-c:v", "ffv1", "-c:a", "pcm_s16le", target_video])
        else:
            extension = Path(video).suffix.lower()
            if extension not in {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"}:
                raise ValueError("请选择 MP4、MOV、MKV、WebM、AVI 或 M4V 视频。")
            target_video = inputs / f"source_{name}{extension}"
            shutil.copy2(video, target_video)
        reference_name = self.prepare_reference(reference)
        actual = probe_video(target_video)
        print(f"片段：从 {start:g}s 开始，约 {selected:.2f}s；原始画幅 {source['width']}×{source['height']}。")
        print("将由工作流转为 24fps、32 像素对齐；有声素材保留原声，无声素材使用静音。")
        if selected > 15:
            print("超过 15 秒：使用约 5 秒窗口自动续接。长片需要更多系统内存，请先检查较短片段的身份和接缝。")
        return dict(video=target_video.name, reference=reference_name,
                    duration=float(selected), source=source, prepared=actual, start=float(start))

    def cancel(self, prompt_id):
        queue = self.request("/queue")
        # Cancel only this controller's job; never blindly interrupt a different job.
        if any(row[1] == prompt_id for row in queue.get("queue_running", [])):
            self.request("/interrupt", {})
        self.request("/queue", {"delete": [prompt_id]})

    def execute(self, graph, *, timeout=7200):
        queue = self.request("/queue")
        if queue.get("queue_running") or queue.get("queue_pending"):
            raise RuntimeError("本服务还有任务在运行，请等待完成，或用停止单元格取消当前任务。")
        run_dir = self.root / "runs" / (time.strftime("%Y%m%d-%H%M%S", time.gmtime()) + "-" + uuid.uuid4().hex[:6])
        run_dir.mkdir(parents=True)
        write_json(run_dir / "prompt.json", graph)
        response = self.request("/prompt", {"prompt": graph, "client_id": uuid.uuid4().hex})
        if response.get("node_errors") or not response.get("prompt_id"):
            raise RuntimeError("工作流校验失败：" + json.dumps(response, ensure_ascii=False)[:6000])
        prompt_id = response["prompt_id"]
        write_json(self.root / "active-job.json", {"prompt_id": prompt_id, "run_dir": str(run_dir)})
        print("任务已提交：", prompt_id, flush=True)
        started, reported = time.monotonic(), 0.0
        try:
            while time.monotonic() - started < timeout:
                history = self.request("/history/" + prompt_id).get(prompt_id)
                if history:
                    write_json(run_dir / "history.json", history)
                    status = history.get("status", {})
                    if status.get("status_str") != "success":
                        message = json.dumps(status.get("messages", status), ensure_ascii=False)
                        raise RuntimeError("生成失败：" + message[-5000:] + "\n日志：" + str(self.log))
                    result = find_saved_video(history, self.comfy / "output")
                    report = probe_video(result, count_frames=True)
                    n, d = report["fps"].split("/")
                    if float(d) == 0 or abs(float(n) / float(d) - 24) > 0.01:
                        raise RuntimeError("生成文件帧率异常，请检查 " + str(result))
                    write_json(run_dir / "result.json", dict(file=str(result), media=report,
                               elapsed_seconds=time.monotonic() - started,
                               comfy_revision=COMFY_REVISION, relay_revision=RELAY_REVISION))
                    print(f"[OK] {report['width']}×{report['height']} / {report['frames']} 帧 / {report['duration']:.2f}s")
                    print("输出：", result)
                    return result
                elapsed = time.monotonic() - started
                if elapsed - reported >= 30:
                    print(f"运行中：{elapsed / 60:.1f} 分钟；首次运行包含模型加载与内核初始化。", flush=True)
                    reported = elapsed
                time.sleep(2)
            raise TimeoutError("任务超过 2 小时，已请求停止。")
        except (KeyboardInterrupt, TimeoutError):
            try:
                self.cancel(prompt_id)
            except Exception:
                print("停止请求未送达；请检查服务状态后使用停止单元格重试。")
            raise
        finally:
            # Keep the ID for the stop cell and post-failure inspection.
            print("本次记录：", run_dir)

    def stop_current_job(self):
        record = self.root / "active-job.json"
        if not record.is_file():
            print("尚未提交任务。")
            return
        self.cancel(json.loads(record.read_text())["prompt_id"])
        print("已请求停止本 notebook 最近提交的任务；模型和素材保留。")


# H3 workflow bundle
import base64, zlib
WORKFLOW_BUNDLE = json.loads(zlib.decompress(base64.b64decode('...')))
'''

exec(compile(RUNTIME_SOURCE, "h3_colab.py", "exec"), globals())
deployment = H3Colab("/content/H3CharacterSwap", port=8188)
(deployment.root / "h3_colab.py").write_text(RUNTIME_SOURCE, encoding="utf-8")
deployment.check_environment()

### 2. 安装 ComfyUI

安装固定版本的 ComfyUI、H3 Relay、独立 Python 环境与 cloudflared，并检查 CUDA/BF16。直接运行，无需填写选项。

In [ ]:
# @title 2. 安装 ComfyUI {display-mode: "form"}
if "deployment" not in globals():
    raise RuntimeError("请先运行第 1 步。")
deployment.install_environment()

### 3. 下载模型与工作流

自动下载 H3 Ref2VA、编码器、VAE、SAM 3.1、Character Swap V1.5 和 FL2V Turbo8，并安装预设工作流。已完成的模型会校验并复用。

通常无需 HF Token；如果你已在 Colab Secrets 中设置 `HF_TOKEN`，本步骤会自动读取。

In [ ]:
# @title 3. 下载模型与工作流 {display-mode: "form"}
if "deployment" not in globals() or not deployment.python.is_file():
    raise RuntimeError("请先运行第 1、2 步。")
hf_token = ""
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN") or ""
except Exception:
    pass
try:
    deployment.prepare_models(token=hf_token, workers=2)
finally:
    hf_token = ""
    deployment.env.pop("HF_TOKEN", None)

### 4. 启动并打开 ComfyUI

与其他 notebook 一样，读取 `CF_TUNNEL_TOKEN`，启动 `8188` 端口并连接 Cloudflare。连接后在下方点击 **“点击打开 ComfyUI”**。若 Colab 的外网探测被返回 401/403，仍会显示链接供浏览器确认，并明确标注外网检查未通过。

进入 ComfyUI 后，在左侧工作流选择 **01_H3_Single_Full_Body**、**02_H3_Face_Only** 或 **03_H3_Multi_People**。01、02 上传原视频和一张参考图；03 在左上角选择 **1–5 人**，等待切换，再为显示的人物 A–E 各上传一张角色图。默认选 2 人。切换人数会保留已上传的图片、人物编号和参数。

默认 `duration_seconds = 0`，按**整段原视频**处理，768 档（≤1MP）、24fps、8 步、CFG 1，并保留原声。

| 工作流 | 参考图 | 默认遮罩与处理 |
| --- | --- | --- |
| 01 单人全换 | 一张清晰全身图 | 人物遮罩扩张 96 像素，匹配整套服装 |
| 02 单人只换脸 | 一张清晰人脸图 | 生成时外扩 8 像素；最终只按原始人脸遮罩向内羽化贴回，不贴回扩张区域；请检查白色区域不包含头发 |
| 03 可选 1–5 人分别替换 | 所选人物各一张全身图 | 在原片上一次固定跟踪，按 A→E 逐人生成、合成，保护其他人的区域 |

**多人映射**：03 板块中每个人物的 `object_indices` 默认依次为 `0–4`，每人填一个不同编号。编号按首帧检测顺序排列，不保证从左到右；先检查各人的白色遮罩，再对应参考图。最多跟踪 5 人，适合选中人物从首帧开始持续可见的连续镜头。跟踪丢失时停止，不按新的检测排序自动换人。遮罩扩张重叠时优先保留前面已生成的区域；人数越多，生成遍数和耗时越多。参考图槽位本身不会自动识别对应原片中的谁。

**完整换装（01 / 03）**：05 板块的提示词要求移除原服装、匹配参考图衣长和裙摆，并重建露出的腿与旧衣服遮住的背景。03 板块的 `H3RelayExpandMask` 默认 `expansion_px = 96`，请检查遮罩覆盖全身和全部原服装。参考图建议使用清晰的单张全身图。遮罩扩张给边缘留出编辑空间，不能保证模型一定完整换装。

**SAM 丢帧补检（01 / 02 单人模式）**：初始检测与视频跟踪阈值均设为 0.20；空遮罩帧自动按 0.35 → 0.20 → 0.10 补检。仍找不到人物时会报出具体帧号和秒数，不跳过视频帧，也不伪造遮罩。单人模式 `recover_empty_frames` 保持开启；03 多人模式使用固定原片跟踪，不使用按序号重新补检的节点。补检修复已经内嵌；旧版需要应用补检修复时，运行第 2 步和第 4 步即可。

**超过 15 秒**：默认无需改时长。工作流按约 5 秒窗口自动续接，保留 18 帧历史。若只想测试片段，01 / 02 在输入板块的 **Prepare Source** 节点修改 `duration_seconds`；03 使用左上角 **时长 0＝整段原视频** 节点。`0` 始终表示整段。当前原生图的整段帧数上限约 149 秒，超长视频会报错，不会悄悄截断。更长的视频请按镜头分段。窗口之间可能出现人物或背景变化，长片还会增加系统内存占用。

In [ ]:
# @title 4. 启动并打开 ComfyUI {display-mode: "form"}
if "deployment" not in globals() or not (deployment.root / "model-manifest.json").is_file():
    raise RuntimeError("请先完成第 1–3 步。")
COMFY_LAUNCHER_SOURCE = r'''"""Start pinned ComfyUI with narrow workflow aliases and Cloudflare compatibility."""
import os
import runpy
import sys
from pathlib import Path
from aiohttp import web


def add_workflow_aliases(manager, routes):
    originals = [route for route in routes
                 if getattr(route, "path", None) == "/userdata/{file}"
                 and getattr(route, "method", None) in {"GET", "POST", "DELETE"}]
    if {route.method for route in originals} != {"GET", "POST", "DELETE"}:
        raise RuntimeError("ComfyUI 工作流访问接口已变化，未安装兼容路由。")
    for route in originals:
        async def workflow_alias(request, handler=route.handler):
            file = request.match_info.get("file", "")
            user_root = manager.get_request_user_filepath(request, None, create_dir=False)
            candidate = manager.get_request_user_filepath(request, file, create_dir=False)
            if not user_root or not candidate:
                return web.Response(status=403)
            workflow_directory = os.path.join(user_root, "workflows")
            if os.path.islink(workflow_directory):
                return web.Response(status=403)
            workflow_root = os.path.realpath(workflow_directory)
            try:
                within = os.path.commonpath((workflow_root, os.path.realpath(candidate))) == workflow_root
            except ValueError:
                within = False
            if not within or os.path.realpath(candidate) == workflow_root:
                return web.Response(status=403)
            return await handler(request)
        routes.route(route.method, "/userdata/{file:workflows/[^/]+}")(workflow_alias)


def install_cf_middleware():
    import middleware.cache_middleware as cm
    orig_cache_control = cm.cache_control

    @web.middleware
    async def cf_safe_middleware(request, handler):
        response = await orig_cache_control(request, handler)
        if request.method == "GET" and request.path == "/" and getattr(response, "status", None) == 200:
            import server
            inst = getattr(server.PromptServer, "instance", None)
            web_root = getattr(inst, "web_root", None) if inst else None
            index_file = Path(web_root) / "index.html" if web_root else None
            if index_file and index_file.is_file():
                html = index_file.read_text(encoding="utf-8", errors="replace")
                html = html.replace("<script ", '<script data-cfasync="false" ')
                new_resp = web.Response(text=html, content_type="text/html")
                for k, v in response.headers.items():
                    if k.lower() not in {"content-length", "content-type", "etag", "last-modified"}:
                        new_resp.headers[k] = v
                new_resp.headers["Cache-Control"] = "no-store, no-transform, must-revalidate"
                response = new_resp
        if isinstance(response, web.StreamResponse) and not isinstance(response, web.WebSocketResponse) and not response.prepared:
            if getattr(response, "status", None) == 200 and (
                request.path.startswith("/assets/")
                or request.path.endswith((".woff2", ".woff", ".ttf"))
            ):
                response.headers["Cache-Control"] = "public, max-age=604800, immutable, no-transform"
            else:
                cc = response.headers.get("Cache-Control", "")
                if "no-transform" not in cc:
                    response.headers["Cache-Control"] = f"{cc}, no-transform" if cc else "no-transform"
            origin = request.headers.get("Origin")
            if origin and response.headers.get("Access-Control-Allow-Origin") == "*":
                response.headers["Access-Control-Allow-Origin"] = origin
                response.headers["Vary"] = "Origin"
        return response

    cm.cache_control = cf_safe_middleware


def main():
    comfy_root = Path("/content/ComfyUI")
    if comfy_root.is_symlink() or not (comfy_root / "main.py").is_file() or not (comfy_root / "comfy/options.py").is_file():
        raise RuntimeError("ComfyUI 源码目录缺失或无效。")
    os.chdir(comfy_root)
    sys.path.insert(0, str(comfy_root))
    sys.argv[0] = str(comfy_root / "main.py")
    import comfy.options
    comfy.options.enable_args_parsing()
    install_cf_middleware()
    from app.user_manager import UserManager
    original = UserManager.add_routes

    def compatible_routes(self, routes):
        original(self, routes)
        add_workflow_aliases(self, routes)

    UserManager.add_routes = compatible_routes
    runpy.run_path("main.py", run_name="__main__")


if __name__ == "__main__":
    main()
'''

deployment.configure_launcher(COMFY_LAUNCHER_SOURCE)
WORKFLOW_BUNDLE = globals().get("WORKFLOW_BUNDLE", {})
RESTART_SOURCE = r'''def can_bind_port(port):
    with socket.socket() as connection:
        connection.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            connection.bind(('127.0.0.1', int(port)))
        except OSError:
            return False
    return True

def start(self):
    self.ensure_relay_sources()
    record = self.root / 'server.json'
    launcher = self.root / 'comfy_launcher.py'
    entry = launcher if launcher.is_file() else self.comfy / 'main.py'
    args = [str(self.python), str(entry), '--listen', '127.0.0.1', '--port', str(self.port), '--disable-auto-launch', '--vram-headroom', '6', '--enable-cors-header', '--enable-compress-response-body', '--disable-all-custom-nodes', '--whitelist-custom-nodes', 'h3-relay']
    signature = hashlib.sha256((sha256_file(entry) + sha256_file(self.relay / 'h3_relay/mask_tools.py')).encode()).hexdigest()
    if record.is_file():
        old = json.loads(record.read_text())
        old_args = old.get('args', [])
        if old_args != args or old.get('entry_sha256') != signature:
            self.stop_owned_server(old)
    try:
        self.request('/system_stats', timeout=3)
        old = json.loads(record.read_text()) if record.exists() else {}
        if old.get('args') != args or not self.owns_process(old.get('pid', 0), [self.comfy / 'main.py', launcher]):
            raise RuntimeError(f'端口 {self.port} 已被其他进程使用，请更改 notebook 的端口。')
        print('复用本 notebook 已启动的 ComfyUI。')
    except (urllib.error.URLError, TimeoutError, ConnectionError):
        if not can_bind_port(self.port):
            raise RuntimeError(f'端口 {self.port} 仍有服务监听；未启动重复服务，请检查现有进程。')
        with self.log.open('a') as log:
            process = subprocess.Popen(args, cwd=self.comfy, env=self.env, stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        write_json(record, {'pid': process.pid, 'args': args, 'entry_sha256': signature})
        deadline = time.monotonic() + 600
        while time.monotonic() < deadline:
            if process.poll() is not None:
                raise RuntimeError('ComfyUI 启动失败：\n' + self.log.read_text(errors='replace')[-6000:])
            try:
                self.request('/system_stats', timeout=3)
                break
            except (urllib.error.URLError, TimeoutError, ConnectionError):
                time.sleep(2)
        else:
            process.terminate()
            raise TimeoutError('ComfyUI 10 分钟内未就绪，请查看 ' + str(self.log))
    info = self.request('/object_info', timeout=120)
    recipes = [profile['api'] for profile in WORKFLOW_BUNDLE.values()] if WORKFLOW_BUNDLE else [self.template()]
    required = {node['class_type'] for recipe in recipes for node in recipe.values()}
    missing = required - set(info)
    if missing:
        raise RuntimeError('缺少工作流节点：' + ', '.join(sorted(missing)) + '\n' + self.log.read_text(errors='replace')[-4000:])
    self.object_info = info

def stop_owned_server(self, record):
    pid = record.get('pid', 0)
    if not self.owns_process(pid, [self.comfy / 'main.py', self.root / 'comfy_launcher.py']):
        return
    args = record.get('args', [])
    try:
        port = int(args[args.index('--port') + 1])
        with urllib.request.urlopen(f'http://127.0.0.1:{port}/queue', timeout=5) as response:
            queue = json.load(response)
    except Exception as error:
        raise RuntimeError('旧服务状态无法确认，请先检查 comfyui.log 后重跑第 4 步。') from error
    if queue.get('queue_running') or queue.get('queue_pending'):
        raise RuntimeError('旧服务仍在执行任务，完成后重跑第 4 步即可更新访问方式。')
    print('正在更新本 notebook 的服务访问配置…', flush=True)
    os.kill(pid, signal.SIGTERM)
    for _ in range(60):
        exited = not self.owns_process(pid, [self.comfy / 'main.py', self.root / 'comfy_launcher.py'])
        if exited and can_bind_port(port):
            return
        time.sleep(0.5)
    raise RuntimeError(f'旧服务或其监听端口 {port} 尚未释放；未启动重复服务，请检查 comfyui.log。')'''

exec(compile(RESTART_SOURCE, "h3_restart.py", "exec"), globals())
deployment.start = start.__get__(deployment, type(deployment))
deployment.stop_owned_server = stop_owned_server.__get__(deployment, type(deployment))
UI_OPEN_SOURCE = r'''def open_ui(self):
    token = os.environ.get('CF_TUNNEL_TOKEN', '').strip()
    if not token:
        try:
            from google.colab import userdata
            token = (userdata.get('CF_TUNNEL_TOKEN') or '').strip()
        except Exception:
            pass
    if not token:
        raise RuntimeError('请与其他 notebook 一样，在 Colab 左侧 🔑 Secrets 设置 CF_TUNNEL_TOKEN 并开启访问权限。')
    token = token.split()[-1]
    self.start()
    binary = self.ensure_cloudflared()
    log_path = self.root / 'cloudflared.log'
    record_path = self.root / 'cloudflared.json'
    if record_path.is_file():
        previous = json.loads(record_path.read_text())
        if self.owns_process(previous.get('pid', 0), [binary]):
            os.kill(previous['pid'], signal.SIGTERM)
    env = dict(self.env, TUNNEL_TOKEN=token)
    with log_path.open('w') as log:
        process = subprocess.Popen([str(binary), 'tunnel', '--no-autoupdate', '--protocol', 'http2', '--edge-ip-version', '4', 'run', '--url', self.base], env=env, stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    token = ''
    env.pop('TUNNEL_TOKEN', None)
    write_json(record_path, {'pid': process.pid})
    print('正在连接 Cloudflare，成功后会显示打开链接…', flush=True)
    deadline = time.monotonic() + 60
    url = None
    routes = []
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError('Cloudflare 连接失败，请检查 CF_TUNNEL_TOKEN。日志：' + str(log_path))
        log = log_path.read_text(errors='replace')
        routes = cloudflare_routes(log)
        url = matching_cloudflare_url(routes, self.port)
        if 'Registered tunnel connection' in log and url:
            break
        time.sleep(1)
    else:
        if routes and (not url):
            raise RuntimeError(f'Cloudflare 域名未指向 {self.base}，请检查 Tunnel 的 Service URL。')
        raise RuntimeError('Cloudflare 未返回可打开的域名，请查看 ' + str(log_path))
    print('[已连接] ComfyUI 访问地址：', url, flush=True)
    access = {'url': url, 'tunnel_connected': True, 'public_verified': False}
    write_json(self.root / 'public-url.json', access)
    probe = urllib.request.Request(url + '/system_stats', headers={'User-Agent': 'H3-Colab/1.0 (+https://github.com/boltguo/colab)', 'Accept': 'application/json'})
    last_error = None
    for attempt in range(3):
        try:
            with urllib.request.urlopen(probe, timeout=5) as response:
                data = json.load(response)
            if not isinstance(data, dict) or 'devices' not in data:
                raise ValueError('域名未返回 ComfyUI 状态。')
            access.update(public_verified=True, http_status=200)
            write_json(self.root / 'public-url.json', access)
            print('[OK] ComfyUI 公网访问正常：', url)
            return url
        except urllib.error.HTTPError as error:
            last_error = f'HTTP {error.code}'
            access['http_status'] = error.code
            if error.code in {401, 403}:
                print(f'[请在浏览器确认] 外网探测返回 HTTP {error.code}，下方仍会显示访问链接。', flush=True)
                break
        except Exception as error:
            last_error = str(error)
        if attempt < 2:
            time.sleep(2)
    access['public_probe_error'] = str(last_error)
    write_json(self.root / 'public-url.json', access)
    print(f'[外网检查未通过] {last_error}。请点击链接确认；若浏览器也报错，请检查 Cloudflare 访问日志。')
    return url'''

exec(compile(UI_OPEN_SOURCE, "h3_open_ui.py", "exec"), globals())
deployment.open_ui = open_ui.__get__(deployment, type(deployment))
ui_url = deployment.open_ui()
from IPython.display import Markdown, display
display(Markdown(f"### [点击打开 ComfyUI]({ui_url})"))
print("进入后选择：01_H3_Single_Full_Body / 02_H3_Face_Only / 03_H3_Multi_People（可选 1–5 人）")

### 可选：在 notebook 里直接上传并生成

已经在 ComfyUI 中操作时可跳过。选择模式，多人模式再选 `PEOPLE_COUNT`（1–5），只会要求上传对应数量的参考图。先显示遮罩预览，再生成和下载 MP4。`CLIP_SECONDS` 默认 `0`，表示处理整段，最多 149 秒；填正数可测试指定秒数。A–E 默认对应首帧跟踪编号 0–4，复杂遮挡请先在 ComfyUI 检查映射和遮罩。

“全部运行”也会执行这个可选单元格并弹出上传框；只部署服务时运行前四步即可。

In [ ]:
# @title 可选：上传视频和角色图，直接生成 {display-mode: "form"}
CLIP_SECONDS = 0.0 # @param {type:"number"}
MODE = "single_full" # @param ["single_full", "face_only", "multi_people"]
PEOPLE_COUNT = "2" # @param ["1", "2", "3", "4", "5"]
if "deployment" not in globals() or not (deployment.root / "model-manifest.json").is_file():
    raise RuntimeError("请先完成前四步部署。")
from google.colab import files
from IPython.display import Video, display

def upload_one(label):
    print(label, flush=True)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("每次请选择一个文件。")
    name, data = next(iter(uploaded.items()))
    folder = deployment.root / "uploads"
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / (uuid.uuid4().hex[:8] + "_" + Path(name).name)
    target.write_bytes(data)
    return str(target)

raw_video_path = upload_one("请选择原视频：")
raw_reference_path = upload_one("请选择清晰人脸参考图：" if MODE == "face_only" else "请选择人物 A 的角色全身参考图：")
media = deployment.prepare_inputs(raw_video_path, raw_reference_path, duration=CLIP_SECONDS)
people_count = int(PEOPLE_COUNT) if MODE == "multi_people" else 1
references = [media["reference"]]
for letter in "BCDE"[:people_count - 1]:
    references.append(deployment.prepare_reference(upload_one("请选择人物 " + letter + " 的角色全身参考图：")))
graph = make_mode_graph(deployment.template(MODE, people_count=people_count), MODE,
                        media["video"], media["reference"], references=references,
                        people_count=people_count, duration=CLIP_SECONDS)
write_json(deployment.root / "selected-inputs.json", dict(media=media, mode=MODE,
           references=references, people_count=people_count, requested_duration=CLIP_SECONDS, seed=42))
for target in "abcde"[:people_count]:
    print("正在生成遮罩预览：" + target.upper())
    mask_video = deployment.execute(preview_graph(graph, target=target))
    display(Video(filename=str(mask_video), embed=True, width=720))
print("正在生成换人视频…")
result_video = deployment.execute(graph)
display(Video(filename=str(result_video), embed=True, width=720))
files.download(str(result_video))

### 说明与排错

- **换人设置**：在 ComfyUI 节点中调整人物描述、选择方式、时长和提示词；部署步骤使用固定预设。想检查多人遮罩，可先运行到工作流的遮罩预览节点。
- **已有部署更新**：先运行本 notebook 第 1 步，再运行下方“更新现有部署”单元格，刷新整个 ComfyUI 页面。已有模型和服务复用；同名工作流有修改时会先备份。多人控件由 notebook 内嵌安装，单独导入 JSON 无法安装控件。
- **下载中断**：重新运行第 3 步，已通过校验的文件会复用。下载或 CUDA 错误详情见安装目录中的 `install.log`、`comfyui.log`。
- **打开界面**：第 4 步输出可点击的 Cloudflare 域名。请确认 `CF_TUNNEL_TOKEN` 已启用，Tunnel 的 Service URL 为 `http://127.0.0.1:8188`。连接日志为安装目录中的 `cloudflared.log`。
- **显存不足 / 跟踪丢失**：在工作流中将分辨率改为 480 档，或换成更短的单镜头素材。
- **找不到人物 / 空遮罩**：01 / 02 单人模式补检开关保持 true，补检修复已内嵌，旧服务用第 2、4 步应用。03 多人模式确认人数不超过画面中的人数、跟踪编号不重复；跟踪丢失时停止，检查遮挡和出画。检测与跟踪阈值均为 0.20。
- **中止 notebook 生成**：点击单元格停止按钮，会请求取消该单元格提交的任务。ComfyUI 的运行任务可在界面中取消。

安装目录：`/content/H3CharacterSwap`；视频输出：`ComfyUI/output/h3-character-swap/`。notebook 直接生成会在 `runs/` 保存 API 工作流与执行记录。保留 6GiB 显存余量，使用独立 Python 3.12、PyTorch 2.11.0 + cu128、ComfyUI `79be670e2d9b`、frontend 1.53.6、comfy-kitchen 0.2.35、H3 Relay 2.0.1。

本地检查覆盖文件格式、工作流及 CPU 素材处理；尚未在你的 96GB GPU 上完成端到端实测。人物身份、遮挡和接续处仍可能漂移，保留原声不代表口型完全一致。

H3 官方的单次输出规格为 4–15 秒。超过 15 秒使用 Relay 的多个生成窗口接续，不代表模型可以一次生成同等长度，也不代表已验证整段长片质量。可先测试 30 秒，再逐步延长；约 149 秒是本工作流帧数限制，不是建议一次跑满的长度。

来源：[V1.5 模型](https://huggingface.co/akatz-ai/MiniMax-H3-Character-Swap-LoRA) · [作者工作流与环境](https://github.com/akatz-ai/h3-relay/blob/e2a7665b6eb8ea37a88c0de2c9c6b69a8f7c9220/example_workflows/toolkit/README.md) · [H3 基座](https://huggingface.co/Comfy-Org/MiniMax-H3) · [Turbo8](https://huggingface.co/lightx2v/Minimax-h3-Turbo) · [SAM 3.1](https://huggingface.co/Comfy-Org/sam3.1)。模型及节点遵循各自上游许可，notebook 未打包权重。

### 可选：更新现有部署

已经在同一台 GPU 上部署过时，先运行本 notebook **第 1 步**，然后只运行本单元格更新工作流和多人控件。无需重跑模型下载或服务启动。刷新整个 ComfyUI 页面后，打开 01 / 02 / 03 工作流即可。

本单元格只写入工作流和界面控件，不重启正在运行的服务；单人 SAM 补检需要更新时，另运行第 2、4 步。

In [ ]:
# @title 可选：更新现有部署（复用模型和服务） {display-mode: "form"}
if "deployment" not in globals() or not WORKFLOW_BUNDLE:
    raise RuntimeError("请先运行本 notebook 的第 1 步。")
if not (deployment.comfy / "main.py").is_file() or not deployment.relay.is_dir():
    raise RuntimeError("未找到现有部署；首次使用请按第 1–4 步完成部署。")
for workflow_path in install_workflow_bundle(deployment.comfy, WORKFLOW_BUNDLE):
    print("[OK] " + Path(workflow_path).name)
print("请刷新整个 ComfyUI 页面，打开 01 / 02 / 03 工作流。")
print("多人可选 1–5 人；默认时长 0＝整段原视频。")